# Regression Models

In [1]:
from protein_benchmark_models.utils import seed_everything

seed = 42
seed_everything(seed)

#data_path = "tape/fluorescence/"
#data_path = "tape/stability/"
data_path = "flip2/amylase/random_split/"
#data_path = "flip2/amylase/by_mutation/"

### Model Registry

In [2]:
from protein_benchmark_models.models import ModelRegistry
print(ModelRegistry.list())

/Users/dylanelliott/workspace/protein-benchmark-models/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['ridge_regressor', 'mlp_regressor', 'cnn_regressor']


### Train Ridge Regressor

In [3]:
import os
import pandas as pd
from protein_benchmark_models.data import OneHotSequenceDataset

# Train dataset
train_data_path = os.path.join(f".data/{data_path}/train.csv")
train_df = pd.read_csv(train_data_path)
sequences = train_df["sequence"].tolist()
targets = train_df["target"].to_numpy()
max_seq_len = train_df["sequence"].map(lambda x: len(x)).max()
train_dataset = OneHotSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Valid dataset
val_data_path = os.path.join(f".data/{data_path}/valid.csv")
val_df = pd.read_csv(val_data_path)
sequences = val_df["sequence"].tolist()
targets = val_df["target"].to_numpy()
val_dataset = OneHotSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Create model
model = ModelRegistry.get("ridge_regressor")(
    seed=seed,
    alpha=1.0
)

# Train, evaluate, and save
model.train(
    experiment_name=f"{data_path.replace('/', '_')}regression",
    run_name=None,
    train_data=train_dataset,
    val_data=val_dataset,
    model_path=f".models/{data_path.replace('/', '_')}ridge_regressor",
    tracking=True
)

[ridge_regressor] Train X: (2372, 9350)
[ridge_regressor] Train y: (2372,)
[ridge_regressor] Training model
[ridge_regressor] Valid RMSE: 0.0386
[ridge_regressor] Valid R2: 0.5139
[ridge_regressor] Valid SpearmanR: 0.6561
🏃 View run useful-penguin-470 at: http://localhost:5000/#/experiments/3/runs/281f6c02b50a402a96ac8666085fab25
🧪 View experiment at: http://localhost:5000/#/experiments/3


### Test Ridge Regressor

In [4]:
import os
import numpy as np
from protein_benchmark_models.data import OneHotSequenceDataset
from protein_benchmark_models.utils import evaluate

# Test dataset
test_data_path = os.path.join(f".data/{data_path}/test.csv")
test_df = pd.read_csv(test_data_path)
sequences = test_df["sequence"].tolist()
targets = test_df["target"].to_numpy()
test_dataset = OneHotSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Evaluate on test set
X = np.stack([test_dataset[i]["one_hots"].numpy().flatten() for i in range(len(test_dataset))])
y = test_dataset.targets.numpy()
metrics = evaluate(model, X, y)

print(f"Test RMSE: {metrics['rmse']:.04f}")
print(f"Test R2: {metrics['r2']:.04f}")
print(f"Test SpearmanR: {metrics['spearmanr']:.04f}")

Test RMSE: 0.0397
Test R2: 0.4967
Test SpearmanR: 0.6757


### Train MLP Regressor

In [5]:
import os
import pandas as pd
from protein_benchmark_models.data import OneHotSequenceDataset

# Train dataset
train_data_path = os.path.join(f".data/{data_path}/train.csv")
train_df = pd.read_csv(train_data_path)
sequences = train_df["sequence"].tolist()
targets = train_df["target"].to_numpy()
max_seq_len = train_df["sequence"].map(lambda x: len(x)).max()
train_dataset = OneHotSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Valid dataset
val_data_path = os.path.join(f".data/{data_path}/valid.csv")
val_df = pd.read_csv(val_data_path)
sequences = val_df["sequence"].tolist()
targets = val_df["target"].to_numpy()
val_dataset = OneHotSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Create model
model = ModelRegistry.get("mlp_regressor")(
    layer_dims = [max_seq_len * len(train_dataset.vocab), 1024, 1],
    hidden_activation = "LeakyReLU",
    output_activation = "Identity",
    use_bias = True,
    norm = "layer",
    seed=seed
)

# Train, evaluate, and save
model.train(
    experiment_name=f"{data_path.replace('/', '_')}regression",
    train_data=train_dataset,
    val_data=val_dataset,
    model_path=f".models/{data_path.replace('/', '_')}mlp_regressor",
    tracking=True,
    lr=1e-4,
    weight_decay = 0.01,
    max_epochs=5,
    batch_size=256,
    val_frequency=1,
    patience=25
)

Epoch: 5/5 | train_loss: 0.0662 | val_loss: 0.0587 | best_val_loss: 0.0587: 100%|██████████| 5/5 [00:09<00:00,  1.83s/it]


[mlp_regressor] Valid RMSE: 0.0585
[mlp_regressor] Valid R2: -0.1156
[mlp_regressor] Valid SpearmanR: 0.4569
🏃 View run selective-lark-659 at: http://localhost:5000/#/experiments/3/runs/a0e33d05fc46421d99b9e4c408cc2d89
🧪 View experiment at: http://localhost:5000/#/experiments/3


### Test MLP Regressor

In [6]:
import os
import numpy as np
from protein_benchmark_models.data import OneHotSequenceDataset
from protein_benchmark_models.utils import evaluate

# Test dataset
test_data_path = os.path.join(f".data/{data_path}/test.csv")
test_df = pd.read_csv(test_data_path)
sequences = test_df["sequence"].tolist()
targets = test_df["target"].to_numpy()
test_dataset = OneHotSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Evaluate on test set
X = np.stack([test_dataset[i]["one_hots"].numpy().flatten() for i in range(len(test_dataset))])
y = test_dataset.targets.numpy()
metrics = evaluate(model, X, y)

print(f"Test RMSE: {metrics['rmse']:.04f}")
print(f"Test R2: {metrics['r2']:.04f}")
print(f"Test SpearmanR: {metrics['spearmanr']:.04f}")

Test RMSE: 0.0600
Test R2: -0.1515
Test SpearmanR: 0.4629


### Train CNN Regressor

In [7]:
import os
import pandas as pd
from protein_benchmark_models.data import TokenizedSequenceDataset

# Train dataset
train_data_path = os.path.join(f".data/{data_path}/train.csv")
train_df = pd.read_csv(train_data_path)
sequences = train_df["sequence"].tolist()
targets = train_df["target"].to_numpy()
max_seq_len = train_df["sequence"].map(lambda x: len(x)).max()
train_dataset = TokenizedSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Valid dataset
val_data_path = os.path.join(f".data/{data_path}/valid.csv")
val_df = pd.read_csv(val_data_path)
sequences = val_df["sequence"].tolist()
targets = val_df["target"].to_numpy()
val_dataset = TokenizedSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Create model
model = ModelRegistry.get("cnn_regressor")(
    embed_dims=[len(train_dataset.vocab), 64],
    kernel_spec=[
        [3, 32, 1],
        [3, 64, 1],
        [3, 128, 2]
    ],
    seq_length=max_seq_len,
    output_dim=1,
    padding_idx=train_dataset.vocab['<PAD>'],
    hidden_activation="LeakyReLU",
    output_activation="Identity",
    use_bias=True,
    norm="layer",
    seed=seed
)

# Train, evaluate, and save
model.train(
    experiment_name=f"{data_path.replace('/', '_')}regression",
    train_data=train_dataset,
    val_data=val_dataset,
    model_path=f".models/{data_path.replace('/', '_')}cnn_regressor",
    tracking=True,
    lr=1e-4,
    weight_decay = 0.01,
    max_epochs=5,
    batch_size=256,
    val_frequency=1,
    patience=25
)

Epoch: 5/5 | train_loss: 0.0853 | val_loss: 0.0876 | best_val_loss: 0.0749: 100%|██████████| 5/5 [00:19<00:00,  3.97s/it]


[cnn_regressor] Valid RMSE: 0.0873
[cnn_regressor] Valid R2: -1.4883
[cnn_regressor] Valid SpearmanR: 0.0397
🏃 View run unequaled-gnat-532 at: http://localhost:5000/#/experiments/3/runs/52feaf1de7c74b30a5541424d118cc5c
🧪 View experiment at: http://localhost:5000/#/experiments/3


### Test CNN Regressor

In [8]:
import os
import numpy as np
from protein_benchmark_models.data import TokenizedSequenceDataset
from protein_benchmark_models.utils import evaluate

# Test dataset
test_data_path = os.path.join(f".data/{data_path}/test.csv")
test_df = pd.read_csv(test_data_path)
sequences = test_df["sequence"].tolist()
targets = test_df["target"].to_numpy()
test_dataset = TokenizedSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Evaluate on test set
X = np.stack([test_dataset[i]["tokens"].numpy() for i in range(len(test_dataset))])
y = test_dataset.targets.numpy()
metrics = evaluate(model, X, y)

print(f"Test RMSE: {metrics['rmse']:.04f}")
print(f"Test R2: {metrics['r2']:.04f}")
print(f"Test SpearmanR: {metrics['spearmanr']:.04f}")

Test RMSE: 0.0896
Test R2: -1.5638
Test SpearmanR: 0.0483
